In [ ]:
import numpy as np
import pandas as pd

from core import (
    compare_table,
    db_connect,
    global_style,
    global_style_loose,
    load_data,
)

ASSET = "XAUUSD"
TIMEFRAME = "D1"

DATA_PATH = f'~/market_data/{ASSET}/{ASSET}_{TIMEFRAME}.csv'
ohlcv_data = load_data(DATA_PATH)

port_value = ohlcv_data["Close"]
log_returns = np.log(port_value / port_value.shift(1)).dropna()

In [ ]:
import plotly.graph_objects as go

# ── metrics ──────────────────────────────────────────────────────────────
mean_val   = log_returns.mean()
std_val    = log_returns.std()
skew_val   = log_returns.skew()
median_val = log_returns.median()

# ── histogram ────────────────────────────────────────────────────────────
fig = go.Figure()

fig.add_trace(go.Histogram(
    x=log_returns.values,
    nbinsx=80,
    marker={
        "color": "#3987e5",
    },
    hovertemplate="Bin: %{x:.4f}<br>Count: %{y}<extra></extra>",
    histnorm="percent"
))

# ── mean / median vertical lines ────────────────────────────────────────
fig.add_vline(x=mean_val,   line_width=2, line_dash="solid", line_color="#d95926")
fig.add_vline(x=median_val, line_width=2, line_dash="solid", line_color="#199e70")

# ── ±1σ center zone lines ───────────────────────────────────────────────
fig.add_vline(x=mean_val - std_val, line_width=1.5, line_dash="dash", line_color="#c98500")
fig.add_vline(x=mean_val + std_val, line_width=1.5, line_dash="dash", line_color="#c98500")

# ── grouped metrics annotation ───────────────────────────────────────────
metrics_text = (
    f"<span style='color:#d95926'>━</span> Mean: {mean_val:.5f}<br>"
    f"<span style='color:#199e70'>━</span> Median: {median_val:.5f}<br>"
    f"<span style='color:#c98500'>╌</span> ±1σ: {std_val:.5f}<br>"
    f"Skew: {skew_val:.4f}"
)

fig.add_annotation(
    xref="x domain",
    yref="y domain",
    x=0.98,
    y=0.95,
    text=metrics_text,
    showarrow=False,
    align="left",
    xanchor="right",
    yanchor="top",
    font={"size": 12, "color": "white", "family": "JetBrainsMono Nerd Font"},
    bordercolor="white",
    borderwidth=1,
    borderpad=6,
    bgcolor="rgba(17, 27, 33, 0.7)",
)

# ── layout ───────────────────────────────────────────────────────────────
fig.update_layout(
    template="plotly_dark",
    paper_bgcolor="#1a1a19",
    plot_bgcolor="#1a1a19",
    title={
        "text": f"{ASSET} {TIMEFRAME} – Log Returns Distribution",
        "font": {"color": "#f0efec", "size": 18},
    },
    xaxis={
        "title": "Log Return",
        "color": "#c3c2b7",
        "gridcolor": "rgba(195,194,183,0.12)",
        "zeroline": False,
    },
    yaxis={
        "title": "Count(%)",
        "color": "#c3c2b7",
        "gridcolor": "rgba(195,194,183,0.12)",
        "zeroline": False,
    },
    font={"family": "JetBrainsMono Nerd Font", "color": "#f0efec"},
    margin={"l": 60, "r": 24, "t": 72, "b": 48},
    bargap=0.2,
    height=500,
    showlegend=False,
)

fig.show()

In [ ]:
def get_dataframe_chunks(df, offset=0, max_chunk_size=1000, step = 1):
    total_rows = len(df)

    # Start the loop at the offset, moving forward by chunk_size each time
    for i in range(offset, total_rows, step):
        end = i + 1
        start = max(0, i - max_chunk_size)
        yield df.iloc[start : end]

chunk_generator = get_dataframe_chunks(log_returns, max_chunk_size=20, step=1)
result = []

for index, chunk in enumerate(chunk_generator, start=1):
    result.append({
      "mean": chunk.mean(),
      "std": chunk.std()
    })

In [ ]:
import plotly.graph_objects as go
import pandas as pd

# Convert the list of dicts `result` into a DataFrame for easier manipulation
# Each entry contains the mean and std of a chunk of log returns.
_df = pd.DataFrame(result)
_df['upper_band'] = _df['mean'] + _df['std']
_df['lower_band'] = _df['mean'] - _df['std']

# ── line chart of mean with ±1σ band ────────────────────────────────────────
fig = go.Figure()

# Mean line
fig.add_trace(go.Scatter(
    x=_df.index + 1,  # chunk index (1‑based for readability)
    y=_df['mean'],
    mode='lines',
    name='Mean',
    line=dict(color='#3987e5'),
))

# Upper band (mean + std)
fig.add_trace(go.Scatter(
    x=_df.index + 1,
    y=_df['upper_band'],
    mode='lines',
    name='Mean + 1σ',
    line=dict(dash='dash', color='#c98500'),
    showlegend=True,
))

# Lower band (mean - std) – fill between upper and lower for a shaded band
fig.add_trace(go.Scatter(
    x=_df.index + 1,
    y=_df['lower_band'],
    mode='lines',
    name='Mean - 1σ',
    line=dict(dash='dash', color='#c98500'),
    fill='tonexty',               # fills between this trace and the previous (upper band)
    fillcolor='rgba(201,133,0,0.2)',
    showlegend=True,
))

# ── layout ───────────────────────────────────────────────────────────────
fig.update_layout(
    template='plotly_dark',
    paper_bgcolor='#1a1a19',
    plot_bgcolor='#1a1a19',
    title={
        'text': f"{ASSET} {TIMEFRAME} – Chunkwise Log Return Mean ±1σ",
        'font': {'color': '#f0efec', 'size': 18},
    },
    xaxis={
        'title': 'Chunk Index',
        'color': '#c3c2b7',
        'gridcolor': 'rgba(195,194,183,0.12)',
        'zeroline': False,
    },
    yaxis={
        'title': 'Log Return',
        'color': '#c3c2b7',
        'gridcolor': 'rgba(195,194,183,0.12)',
        'zeroline': False,
    },
    font={'family': 'JetBrainsMono Nerd Font', 'color': '#f0efec'},
    margin={'l': 60, 'r': 24, 't': 72, 'b': 48},
    height=500,
    showlegend=True,
)

fig.show()